# Backend en Kaggle

Inicia `llama-server` con un modelo de HuggingFace y lo expone mediante un túnel,
para que el intermediario (en la computadora local) lo use como un backend más.

**Antes de ejecutar:** en el panel de la derecha, `Accelerator` = GPU (T4 x2 o P100)
e `Internet` = On. Sin internet no se descarga ni el modelo ni el túnel.

**Límites:** 12 h por sesión, 30 h por semana. La sesión se cierra sola si la
pestaña queda cerrada mucho tiempo.

**Seguridad:** la URL del túnel es pública y sin contraseña. Cualquiera que la
tenga puede usar el modelo, y por ahí viaja el contenido de los archivos.
Es aceptable para una prueba; conviene pensarlo dos veces con código importante.

In [ ]:
# 1. Que GPU se asigno
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. El motor

llama.cpp NO publica binarios de Linux, así que hay que compilar (~15-25 min).
Se compila UNA vez: después de esta celda, usa `Save Version` y en la próxima sesión
añade la salida como dataset de entrada — así inicia en segundos.

Se usa **upstream** (`ggml-org/llama.cpp`), que es el mismo motor que
`backends/linux/bin-spark` en la computadora local. Eso permite comparar contra una
medición local con el MISMO motor, y que la única diferencia sea el hardware.

In [ ]:
import os, glob, subprocess, pathlib

# El motor. llama.cpp NO publica binarios de Linux, hay que compilar (~20 min).
# Con `Files only` en PERSISTENCE, el resultado sobrevive a la sesion y las
# siguientes inician en segundos.
BIN = None
for cand in ("/kaggle/input", "/kaggle/working"):
    for q in pathlib.Path(cand).rglob("llama-server"):
        BIN = q.parent; break
    if BIN: break

if BIN:
    print("motor ya disponible:", BIN)
else:
    # cmake no encuentra CUDA::cuda_driver porque libcuda.so NO esta ni en
    # lib64 ni en stubs: en Kaggle esta en /usr/local/cuda/compat/. Descubierto
    # el 11/09/2026 despues de que fallara la receta estandar.
    cands = []
    for raiz in ("/usr/local/cuda", "/usr/lib/x86_64-linux-gnu", "/usr/local/nvidia"):
        cands += glob.glob(raiz + "/**/libcuda.so*", recursive=True)
    real = next((c for c in cands if c.endswith("libcuda.so")), None) or (cands[0] if cands else None)
    print("libcuda para enlazar:", real)
    os.makedirs("/kaggle/working/cudastub", exist_ok=True)
    if real and not os.path.exists("/kaggle/working/cudastub/libcuda.so"):
        os.symlink(real, "/kaggle/working/cudastub/libcuda.so")

    def ejecutar(cmd, log):
        with open(log, "w") as f:
            p = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
        print("OK" if p.returncode == 0 else f"FALLO ({p.returncode})")
        if p.returncode: print(open(log).read()[-1500:])
        return p.returncode == 0

    subprocess.run("git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp", shell=True)
    print("configurando...")
    if ejecutar("cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=ON "
              "-DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_NATIVE=ON -DGGML_CCACHE=OFF "
              "-DCMAKE_LIBRARY_PATH=/kaggle/working/cudastub", "/tmp/cmake.log"):
        print("compilando, 15-25 min sin imprimir nada...")
        ejecutar("cd /kaggle/working/llama.cpp && cmake --build build -j$(nproc)", "/tmp/build.log")
    BIN = pathlib.Path("/kaggle/working/llama.cpp/build/bin")

print("llama-server listo:", (BIN / "llama-server").exists())


## 3. El modelo

El modelo se elige en la celda de abajo con `REPO` y `ARCHIVO` (y `BORRADOR` si
tiene cabeza MTP). Hoy está cargado **gemma-4-26B-A4B**; los medidos hasta ahora
están en `pruebas/resultados/resultado_kaggle_harness.md`.

La PRIMERA vez conviene usar un modelo que ya esté medido localmente: es el
CONTROL. Si en Kaggle se comporta igual que en local, el entorno queda validado y
solo entonces conviene probar uno más grande. Cambiar hardware y modelo a la vez
no permite saber a cuál atribuir la diferencia.

Con **T4 x2** hay ~30 GB: caben modelos de hasta ~26 GB con contexto de 65.536.
Con **P100** son 16 GB, y los modelos grandes no caben.

In [ ]:
import os, subprocess, shutil
from huggingface_hub import hf_hub_download

# /kaggle/working tiene 20 GB de tope y `local_dir` guarda el archivo POR
# DUPLICADO (copia + cache): un modelo de 5 GB ocupaba 19. Se usa la cache por
# defecto, que esta en el overlay con 1 TB libre.
subprocess.run("pkill -x llama-server", shell=True)
shutil.rmtree("/root/.cache/huggingface/hub", ignore_errors=True)   # el modelo anterior

REPO     = "unsloth/gemma-4-26B-A4B-it-GGUF"
ARCHIVO  = "gemma-4-26B-A4B-it-UD-Q5_K_M.gguf"
BORRADOR = "MTP/mtp-gemma-4-26B-A4B-it-Q8_0.gguf"      # +50% de velocidad, salida igual
CTX      = 65536                                        # Hermes exige >= 64.000
CACHE_K  = CACHE_V = "q4_0"

MODELO = hf_hub_download(repo_id=REPO, filename=ARCHIVO)
MTP    = hf_hub_download(repo_id=REPO, filename=BORRADOR)
print(f"modelo   {os.path.getsize(MODELO)/2**30:.2f} GB")
print(f"borrador {os.path.getsize(MTP)/2**30:.2f} GB")


# ── Verificar que el GGUF se descargo entero ────────────────────────────────────────
# POR QUE (12/09/2026): K2-MoVA-36B-A4B fallaba con
#   `ggml_cuda_compute_forward: CLAMP failed / illegal memory access`
# a los pocos minutos, y al dia siguiente obtuvo 5/5 con la MISMA config y el
# binario sin recompilar. Un GGUF truncado produce exactamente ese sintoma --
# el kernel lee fuera de rango-- y la vispera el disco se habia llenado al 100%
# por las copias dobles de `local_dir`. Sin esta comprobacion no hay forma de
# distinguir "archivo dañado" de "bug del motor", que es donde quedo detenida la
# medicion. Se compara contra el sha256 que publica el propio Hugging Face, asi
# que verifica AHORA en vez de dejar un numero para comparar despues.
import hashlib
from huggingface_hub import HfApi

def verificar(ruta, repo, archivo):
    try:
        info = HfApi().get_paths_info(repo_id=repo, paths=[archivo])[0]
        esperado = info.lfs.sha256 if getattr(info, "lfs", None) else None
    except Exception as e:
        esperado = None
        print(f"  (no se pudo obtener el sha de HF: {type(e).__name__})")
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(16 << 20), b""):
            h.update(bloque)
    local = h.hexdigest()
    if esperado is None:
        print(f"  {archivo}\n    sha256 local {local}  (sin referencia para comparar)")
        return None
    ok = local == esperado
    print(f"  {'OK ' if ok else 'MAL'} {archivo}\n    {local}")
    if not ok:
        print(f"    ESPERADO {esperado}\n    EL ARCHIVO NO COINCIDE: volver a descargarlo antes de medir nada.")
    return ok

print("\nverificando (tarda ~1 min por cada 20 GB)...")
verificar(MODELO, REPO, ARCHIVO)
verificar(MTP, REPO, BORRADOR)


In [ ]:
import subprocess, time, urllib.request, os

subprocess.run("pkill -x llama-server", shell=True)   # -x: nombre EXACTO, nunca -f
time.sleep(3)

# DOS rutas distintas de libcuda, y confundirlas cuesta la tarde:
#   para COMPILAR  /usr/local/cuda/compat/libcuda.so     (570.x, vieja)
#   para EJECUTAR  /usr/local/nvidia/lib64/libcuda.so.1  (el driver real, 580.x)
# Con la de compat, CUDA no se inicializa y el modelo se ejecuta en CPU a 3 tok/s
# con la GPU vacia, indicando "servidor activo" como si nada.
env = dict(os.environ)
env["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + str(BIN) + ":" + env.get("LD_LIBRARY_PATH", "")

extra = ""
if "MTP" in dir():
    ayuda = subprocess.run(f"{BIN}/llama-server --help", shell=True, capture_output=True, text=True).stdout
    extra = f" --model-draft {MTP}"
    if "-ngld" in ayuda: extra += " -ngld 99"
    if "--spec-draft-n-max" in ayuda: extra += " --spec-draft-n-max 4"
    print("borrador:", extra)

base = f"--model {MODELO} --ctx-size {CTX} --n-predict 32768 --host 127.0.0.1 --port 8080"
base += " --jinja --n-gpu-layers 99 --parallel 4 --kv-unified --flash-attn auto --cont-batching"
base += f" --cache-type-k {CACHE_K} --cache-type-v {CACHE_V} --threads {os.cpu_count()}"
base += f" --temp 0.6 --top-p 0.95 --top-k 20{extra}"

srv = subprocess.Popen([str(BIN / "llama-server")] + base.split(), stdout=open("/tmp/srv.log", "w"), stderr=subprocess.STDOUT, env=env)

for i in range(180):
    if srv.poll() is not None: break
    try: urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2); break
    except Exception: time.sleep(5)

print("el servidor termino" if srv.poll() is not None else "servidor activo")
print(subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("grep -iE 'draft|error' /tmp/srv.log | head -6", shell=True, capture_output=True, text=True).stdout)


## 5. El túnel

`cloudflared` en modo rápido: no pide cuenta ni token, y asigna una URL al azar.
Por eso mismo la URL es lo único que protege al servidor: no conviene publicarla.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
import subprocess, re, time
tun = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8080"],
                       stdout=open("/tmp/tun.log","w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/tun.log").read())
    if m: URL = m.group(0); break
print("URL:", URL or "no aparecio -- revisar /tmp/tun.log")
print()
print("Configurar esa URL en el intermediario como un backend compatible con OpenAI.")

In [ ]:
# 6. Mantener activa la sesion y ver que llega. Dejar en ejecucion.
import time, datetime
while True:
    n = sum(1 for l in open("/tmp/srv.log") if "POST /v1/chat/completions" in l)
    print(f"{datetime.datetime.now():%H:%M}  peticiones atendidas: {n}", flush=True)
    time.sleep(300)